# Assgn_012 — ZeRO / Data Parallel Simulation

ERA V5 Session 12. This notebook wraps `src.pipeline.zero_lab.run_all` and displays the PRIMARY graphics.

Run the lab from the repo root, or execute the cells below.

```text
python tests/assgn012_zero_lab.py
torchrun --nproc_per_node=32 -m tests.assgn012_zero_lab
```

Colab setup — upload Assign_0012_colab.zip to /content first, then uncomment:

```text
# !unzip -q -o /content/Assign_0012_colab.zip -d /content/Assign_0012
# %cd /content/Assign_0012
```

Open notebook: `/content/Assign_0012/notebooks/assgn012_zero_sim.ipynb`

Forbidden: relative paths like `content/Assign_0012` (creates `/content/content/...`).


In [ ]:
from pathlib import Path
import sys
from IPython.display import Image, display, Markdown

ROOT = Path.cwd()
if not (ROOT / "src").is_dir():
    ROOT = Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.pipeline.zero_lab import run_all, format_gate_print

results = run_all(ROOT / "reports")

# Always show boolean + companion values (never a naked True alone).
print("\n".join(format_gate_print(results)))

smoke = results.get("task1_smoke", {})
dp = results.get("task2_dp", {})
zero = results.get("task3_zero", {})
gfx = results.get("task4_graphics", {})
jc = results.get("task_json_compare", {})
primary_n = sum(1 for k in ("ladder", "dashboard", "compute_comm") if gfx.get(k) is not None)

table = f"""
| Gate | ok | Companion values |
| :--- | :---: | :--- |
| task1_smoke | `{smoke.get('ok')}` | smoke_sum={smoke.get('value')} · world_size={smoke.get('world_size')} · backend={smoke.get('backend')} |
| task2_dp | `{dp.get('ok')}` | N_demo={dp.get('n_params')} · max_abs_diff={dp.get('max_abs_param_diff_vs_rank0')} · peak_rss_mb={dp.get('peak_rss_mb')} |
| task3_zero | `{zero.get('ok')}` | mismatches={len(zero.get('mismatches') or [])} · W32_GiB=447.0/122.2/68.1/14.0 · floor_gib={zero.get('floor_gib')} |
| task4_graphics | `{gfx.get('ok')}` | primary_pngs={primary_n} |
| task_json_compare | `{jc.get('ok')}` | stages_pass={jc.get('n_pass')}/{jc.get('n_total')} · report=`{jc.get('report_path')}` |
| **all_ok** | `{results.get('all_ok')}` | PASS if True; FAIL if False · manifest=`{results.get('manifest_path')}` |
"""
display(Markdown(table))


In [ ]:
reports = ROOT / "reports"
for name in [
    "assgn012_memory_ladder.png",
    "assgn012_w32_dashboard.png",
    "assgn012_compute_comm.png",
    "assgn012_activation_vs_seq.png",
]:
    p = reports / name
    display(Markdown(f"### {name}"))
    if p.is_file():
        display(Image(filename=str(p)))
    else:
        print("missing", p)

In [ ]:
from src.utils.zero_accounting import memory_ladder, ladder_as_dicts

rows = [r for r in ladder_as_dicts(memory_ladder()) if r["world_size"] == 32]
print(f"{'stage':8} {'bpp':>8} {'GiB':>8} {'comm':>4} {'fit':>10}")
for r in rows:
    print(f"{r['stage']:8} {r['bytes_per_param']:8.4f} {r['gib_per_gpu']:8.2f} {r['comm']:>4} {r['fit']:>10}")

## Takeaways

- DP and ZeRO-1 **never** fit a 30B model on 80 GB cards (replicated 4 B/weight floor).
- ZeRO-2 fits from **32** GPUs; ZeRO-3 from **8**.
- ZeRO-3 costs **3P** communication; stages 1–2 stay at **2P**.
- Long context activations sit on top of the static ladder — Z2@32 headroom is thin.
